%md
#Gold Layer — KPI Tables & Business Aggregations
### Notebook: 03_gold_kpi_tables
**Pipeline:** Bank Customer Churn Analytics
**Author:** Harshkumar Patel
**Date:** 03/07/2026

---

## What this notebook does
Gold layer transforms the cleaned silver star schema into business-ready 
KPI tables using Spark SQL aggregations for Power BI dashboards

## Gold Layer Rule
> Aggregate it. Summarise it. Make it business ready.

## Source
- **Database:** churn_silver
- **Tables:** fact_churn_events, dim_customer, dim_scores, dim_customer_features
- **Rows:** 80,000

## Target Tables
- **gold_churn_summary** — overall churn KPIs for executive dashboard
- **gold_churn_by_segment** — churn rate by province, age group, balance tier and engagement tier
- **gold_churn_risk_analysis** — validates churn_risk_label against actual churn rates
- **gold_churn_by_occupation** — unique insight into churn rate by customer occupation

## What we do in this notebook
1. Read from Silver star schema tables
2. Join fact and dimension tables
3. Write Spark SQL KPI queries
4. Save aggregated results as Gold Delta tables
5. Validate outputs

# 1. Read from Silver star schema tables

In [0]:
# loading data from the silver layer star schema tables into gold for aggregation
df_fact = spark.table("churn_silver.fact_churn_events")
df_customer = spark.table("churn_silver.dim_customer")
df_scores = spark.table("churn_silver.dim_scores")
df_features = spark.table("churn_silver.dim_customer_features")

print(f"fact rows: {df_fact.count():,}")

%md
## Master View — vw_churn_master
Joining all 4 silver tables into a single view so all Gold KPI queries 
run against one combined source instead of rewriting joins every time.

In [0]:
%sql
CREATE OR REPLACE VIEW churn_silver.vw_churn_master AS
SELECT
    f.*,
    c.credit_score,
    c.gender,
    c.age,
    c.occupation,
    c.origin_province,
    c.customer_segment,
    s.engagement_score,
    s.risk_score,
    ft.age_group,
    ft.balance_tier,
    ft.tenure_group,
    ft.engagement_tier,
    ft.churn_risk_label,
    ft.churn_reason_inferred
FROM churn_silver.fact_churn_events f
JOIN churn_silver.dim_customer c ON f.id = c.id
JOIN churn_silver.dim_scores s ON f.id = s.id
JOIN churn_silver.dim_customer_features ft ON f.id = ft.id

In [0]:
%sql
SELECT * FROM churn_silver.vw_churn_master
LIMIT 5


%md
## gold_churn_summary — Executive KPI Table
Aggregates the entire customer base into a single row of key business metrics.
This table powers the KPI cards in the Power BI dashboard.

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_summary
USING DELTA
AS
SELECT
    COUNT(*)                                                         AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)                    AS total_churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(balance), 2)                                           AS avg_balance,
    ROUND(AVG(engagement_score), 2)                                  AS avg_engagement_score,
    ROUND(AVG(risk_score), 2)                                        AS avg_risk_score
FROM churn_silver.vw_churn_master

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_summary

In [0]:
%sql
SELECT churn_reason_inferred, COUNT(*) 
FROM churn_silver.vw_churn_master 
GROUP BY churn_reason_inferred

gold_churn_by_segment

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_by_segment
USING DELTA
AS

WITH age_segment AS (
    SELECT
        'age_group'                AS segment_type,
        age_group                  AS segment_value,
        COUNT(*)                   AS total_customers,
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
        ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
    FROM churn_silver.vw_churn_master
    GROUP BY age_group
),
balance_segment AS (
    SELECT
        'balance_tier',
        balance_tier,
        COUNT(*),
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END),
        ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
    FROM churn_silver.vw_churn_master
    GROUP BY balance_tier
),
engagement_segment AS (
    SELECT
        'engagement_tier', engagement_tier,
        COUNT(*),
        SUM(CASE WHEN exit = true THEN 1 ELSE 0 END),
        ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)
    FROM churn_silver.vw_churn_master
    GROUP BY engagement_tier
)

SELECT * FROM age_segment
UNION ALL
SELECT * FROM balance_segment
UNION ALL
SELECT * FROM engagement_segment

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_by_segment
ORDER BY churn_rate_pct DESC

gold_churn_by_reason

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_by_reason
USING DELTA
AS
SELECT
    churn_reason_inferred,
    COUNT(*)        AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) AS churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct
FROM churn_silver.vw_churn_master
GROUP BY churn_reason_inferred
ORDER BY churned DESC

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_by_reason
ORDER BY churned DESC

gold_churn_risk_analysis

In [0]:
%sql
CREATE OR REPLACE TABLE churn_gold.gold_churn_risk_analysis
USING DELTA
AS
SELECT
    churn_risk_label,
    COUNT(*)                                                           AS total_customers,
    SUM(CASE WHEN exit = true THEN 1 ELSE 0 END)                     AS churned,
    ROUND(SUM(CASE WHEN exit = true THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS churn_rate_pct,
    ROUND(AVG(engagement_score), 2)                                   AS avg_engagement_score,
    ROUND(AVG(risk_score), 2)                                         AS avg_risk_score
FROM churn_silver.vw_churn_master
GROUP BY churn_risk_label
ORDER BY churn_rate_pct DESC

In [0]:
%sql
SELECT * FROM churn_gold.gold_churn_risk_analysis
ORDER BY churned DESC